# Risk Monitor - EDA and Data Cleaning

This notebook explores the SQLite dataset provided for the Sharesub case study.

Goals:
- understand table structure and relationships
- identify anomalies and data quality issues
- document cleaning decisions
- prepare a reproducible basis for subscriber risk scoring

In [2]:
import sqlite3
from pathlib import Path

import pandas as pd

In [3]:
DB_PATH = Path("../data/raw/risk_monitor_dataset.sqlite")

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

tables

,name
0,complaints
1,memberships
2,payments
3,subscriptions
4,users


In [4]:
users = pd.read_sql_query("SELECT * FROM users", conn)
subscriptions = pd.read_sql_query("SELECT * FROM subscriptions", conn)
memberships = pd.read_sql_query("SELECT * FROM memberships", conn)
payments = pd.read_sql_query("SELECT * FROM payments", conn)
complaints = pd.read_sql_query("SELECT * FROM complaints", conn)

In [5]:
{
    "users": users.shape,
    "subscriptions": subscriptions.shape,
    "memberships": memberships.shape,
    "payments": payments.shape,
    "complaints": complaints.shape,
}

{'users': (2001, 8),
 'subscriptions': (400, 8),
 'memberships': (1083, 7),
 'payments': (7277, 10),
 'complaints': (1213, 9)}

In [6]:
users.head(), subscriptions.head(), memberships.head(), payments.head(), complaints.head()

(   id               email country           signup_date  status  \
 0   1     user_1@yahoo.fr      IT   2021-02-09 21:26:47     1.0   
 1   2    user_2@gmail.com      CH  2022-01-09T06:49:44Z     0.0   
 2   3  user_3@outlook.com      BE            1584765297     1.0   
 3   4  user_4@hotmail.com      DE  2020-07-13T23:53:58Z     0.0   
 4   5  user_5@outlook.com      DE      21/07/2020 08:54     0.0   
 
              last_seen referral_code phone_prefix  
 0  2021-04-28 14:23:00           NaN          +49  
 1  2023-01-05 17:03:20           NaN          +41  
 2  2021-07-30 05:38:37      27cb14dc          +32  
 3  2024-06-10 09:06:04           NaN          +49  
 4  2026-03-06 20:16:50           NaN          +49  ,
    id          brand  owner_id           created_at  status  max_slots  \
 0   1  Microsoft 365         1  2023-01-14 23:23:57       0          4   
 1   2        HBO Max      1670  2023-07-16 21:39:51       2          4   
 2   3   ChatGPT Plus       558  2024-10-02 13

In [7]:
def missing_summary(df, name):
    summary = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean() * 100).round(2).values
    }).sort_values("missing_pct", ascending=False)
    summary["table"] = name
    return summary[["table", "column", "missing_count", "missing_pct"]]

missing = pd.concat([
    missing_summary(users, "users"),
    missing_summary(subscriptions, "subscriptions"),
    missing_summary(memberships, "memberships"),
    missing_summary(payments, "payments"),
    missing_summary(complaints, "complaints"),
])

missing[missing["missing_count"] > 0]

,table,column,missing_count,missing_pct
6,users,referral_code,1418,70.86
7,users,phone_prefix,419,20.94
4,users,status,25,1.25
6,memberships,reason,706,65.19
5,memberships,left_at,637,58.82
9,payments,stripe_error_code,5444,74.81
7,payments,captured_at,3370,46.31
8,complaints,resolution,765,63.07
7,complaints,resolved_at,700,57.71


In [8]:
for col in ["status", "country", "phone_prefix"]:
    print(f"\nusers.{col}")
    print(users[col].value_counts(dropna=False).head(20))

for col in ["status", "brand", "currency"]:
    print(f"\nsubscriptions.{col}")
    print(subscriptions[col].value_counts(dropna=False).head(20))

for col in ["status", "reason"]:
    print(f"\nmemberships.{col}")
    print(memberships[col].value_counts(dropna=False).head(20))

for col in ["status", "currency", "stripe_error_code"]:
    print(f"\npayments.{col}")
    print(payments[col].value_counts(dropna=False).head(20))

for col in ["status", "type", "resolution"]:
    print(f"\ncomplaints.{col}")
    print(complaints[col].value_counts(dropna=False).head(20))


users.status
status
 0.0     1325
 4.0      198
 1.0      179
 3.0      110
 2.0      107
-1.0       38
 NaN       25
 99.0      19
Name: count, dtype: int64

users.country
country
ES        165
AT        153
DE        149
IT        140
BE        135
France    134
fr        134
PT        133
          132
Fra       127
FR        127
DE        122
NL        121
 ES       119
CH        110
Name: count, dtype: int64

users.phone_prefix
phone_prefix
NaN     419
+34     205
+43     197
+39     188
+351    180
+33     170
+49     164
+32     164
+41     158
+31     156
Name: count, dtype: int64

subscriptions.status
status
0    229
2     68
3     57
1     46
Name: count, dtype: int64

subscriptions.brand
brand
Midjourney         36
Microsoft 365      33
HBO Max            31
Disney+            31
Spotify            30
ChatGPT Plus       28
Apple Music        28
NordVPN            25
Notion             25
Adobe CC           24
Netflix            23
Canal+             23
YouTube Premium    22

## First observed anomalies

Main issues identified so far:

- undocumented numeric status codes in `users`, `subscriptions`, `memberships`
- inconsistent country labels in `users.country`
- inconsistent currencies in `payments.currency` and `subscriptions.currency`
- inconsistent payment statuses, including typos and capitalization differences
- inconsistent complaint types and statuses, mixing English, French, uppercase and lowercase
- multiple datetime formats across tables
- missing values in several fields, especially operational fields such as complaint resolution and payment capture time

In [9]:
users[["signup_date", "last_seen"]].head(10), payments[["created_at", "captured_at"]].head(10)

(            signup_date            last_seen
 0   2021-02-09 21:26:47  2021-04-28 14:23:00
 1  2022-01-09T06:49:44Z  2023-01-05 17:03:20
 2            1584765297  2021-07-30 05:38:37
 3  2020-07-13T23:53:58Z  2024-06-10 09:06:04
 4      21/07/2020 08:54  2026-03-06 20:16:50
 5   2023-03-09 18:54:13  2024-01-29 05:53:30
 6  2024-11-25T09:10:59Z  2025-05-14 23:06:23
 7   2022-11-03 16:05:01  2023-09-29 23:22:32
 8  2025-02-15T23:22:07Z  2025-12-23 04:44:01
 9  2025-03-26T08:39:40Z  2025-09-24 03:12:08,
                   created_at          captured_at
 0  2024-12-04T21:23:01+02:00  2024-12-07 14:23:01
 1  2025-01-03T21:23:01+01:00  2025-01-05 11:23:01
 2        2025-02-07 21:23:01  2025-02-10 04:23:01
 3        2025-03-04 21:23:01  2025-03-06 20:23:01
 4        2025-04-03 21:23:01  2025-04-04 23:23:01
 5        2025-05-04 21:23:01  2025-05-05 09:23:01
 6        2024-09-20 02:25:46                  NaN
 7  2024-10-21T02:25:46+02:00                  NaN
 8        2024-11-21 02:25:46     

In [10]:
def clean_text_series(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
         .str.lower()
    )

users["country_clean_preview"] = clean_text_series(users["country"])
payments["status_clean_preview"] = clean_text_series(payments["status"])
payments["currency_clean_preview"] = clean_text_series(payments["currency"])
complaints["status_clean_preview"] = clean_text_series(complaints["status"])
complaints["type_clean_preview"] = clean_text_series(complaints["type"])

print("users.country cleaned preview")
print(users["country_clean_preview"].value_counts(dropna=False).head(20))

print("\npayments.status cleaned preview")
print(payments["status_clean_preview"].value_counts(dropna=False).head(20))

print("\ncomplaints.type cleaned preview")
print(complaints["type_clean_preview"].value_counts(dropna=False).head(20))

users.country cleaned preview
country_clean_preview
es        284
de        271
fr        261
at        153
it        140
be        135
france    134
pt        133
          132
fra       127
nl        121
ch        110
Name: count, dtype: int64

payments.status cleaned preview
status_clean_preview
succeeded    3554
failed       1984
pending       581
refunded      375
disputed      216
success       216
suceeded      210
canceled      141
Name: count, dtype: int64

complaints.type cleaned preview
type_clean_preview
access_denied            269
other                    153
wrong_credentials        145
subscription_inactive    143
owner_unresponsive       142
billing_issue            127
fraud_suspicion          118
accès refusé             116
Name: count, dtype: int64


## Hypotheses on numeric status codes

The exact mapping is not documented, so no definitive assumption should be made yet.

However, the following observations suggest that:
- `users.status` likely mixes active / inactive / blocked-like states, plus anomalous values such as `99` and `-1`
- `subscriptions.status` likely represents lifecycle states such as active / paused / closed / suspended
- `memberships.status` likely reflects membership lifecycle states, potentially including active, left, removed, payment issue, fraud-related removal, etc.

These mappings should be inferred later by cross-checking:
- whether `left_at` is filled
- whether `reason` indicates fraud / payment failure / owner request
- whether payments continue after a given status

In [11]:
memberships.groupby(["status", "reason"], dropna=False).size().reset_index(name="count").sort_values(
    ["status", "count"], ascending=[True, False]
).head(30)

,status,reason,count
0,0,NaN,59
1,1,NaN,428
2,2,,41
5,2,owner_request,35
8,2,NaN,34
3,2,fraud,30
7,2,voluntary,30
6,2,payment_failed,26
4,2,inactive,22
12,3,owner_request,18


In [12]:
memberships.assign(has_left_at=memberships["left_at"].notna()) \
    .groupby(["status", "has_left_at"]) \
    .size() \
    .reset_index(name="count") \
    .sort_values(["status", "count"], ascending=[True, False])

,status,has_left_at,count
0,0,False,59
1,1,False,414
2,1,True,14
4,2,True,202
3,2,False,16
5,3,True,107
6,4,False,64
7,5,False,84
8,6,True,68
9,7,True,55


## Preliminary cleaning decisions

At this stage, I would apply the following principles:

- standardize text categories by trimming spaces and lowercasing values
- normalize equivalent categories:
  - `succeeded`, `Succeeded`, `success`, `suceeded` -> `succeeded`
  - `failed`, `FAILED` -> `failed`
  - `open`, `Open` -> `open`
  - `resolved`, `RESOLVED` -> `resolved`
  - `ACCESS_DENIED`, `access_denied`, `Accès refusé` -> `access_denied`
- normalize currencies:
  - `EUR`, `eur`, `€` -> `EUR`
- normalize countries where possible:
  - `fr`, `Fra`, `France` -> `FR`
  - trim spaces in `DE ` and ` ES`
- keep undocumented numeric status codes as-is for now and reverse-engineer them later using behavioral patterns
- keep missingness when it may carry business meaning instead of dropping rows too early

## Next steps for scoring

The next phase will focus on:

- building a cleaned analytical layer from the raw tables
- engineering subscriber-level features
- defining a deterministic and explainable risk score
- handling edge cases such as new subscribers, low-history subscribers, and inactive users